In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models
import glob
import hashlib
from tqdm import tqdm

from sklearn.model_selection import (
    GroupKFold, GridSearchCV,
    cross_val_predict, cross_validate,
)
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
import sklearn

from sklearn.metrics import (
    mean_absolute_error, r2_score,
    mean_squared_error, root_mean_squared_error,
)
from matplotlib import pyplot as plt


## Constants

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_CACHE_DIR = Path("embeddings") # For caching image embeddings
EMBEDDING_CACHE_DIR.mkdir(exist_ok=True)
CSV_PATH = Path("experiments/experiments_1_36.csv")
TOP_FOLDER = Path("top_view_images")
SIDE_FOLDER = Path("side_view_images")

OUTER_SPLITS = 5   # outer CV  =  performance estimate
INNER_SPLITS = 5   # inner CV  =  hyperparameter selection
RANDOM_STATE = 42

## Data Loading

In [3]:
data = pd.read_csv(CSV_PATH)

samples = data.copy()
samples = samples[samples["volume"] > 0]
samples = samples[samples["pic_top"].notna() & samples["pic_side"].notna()]

samples["exp_id"] = samples["exp_id"].astype(int)
samples["pic_top"] = samples["pic_top"].astype(int)
samples["pic_side"] = samples["pic_side"].astype(int)

print("Samples:", len(samples))
display(samples.head())

Samples: 140


,file_name,bridge_angle,Fr,Fr_ini,plates,cubes,logs,Q[dm],H_up,H_down,...,H_down_corr,pic_side,pic_top,dh,dh_d0,rel_BW,carpet_length_avg,carpet_length_max,exp_id,volume
2,NaN,22.5,0.018605,0.55,0.25,0.00,0.75,18.6,142.5,14.10,...,87.0,161,72,49.5,0.550000,NaN,NaN,NaN,1,38
3,NaN,22.5,0.018605,0.55,0.25,0.00,0.75,18.6,145.5,14.10,...,87.0,163,73,52.5,0.583333,NaN,NaN,NaN,1,57
4,NaN,22.5,0.018605,0.55,0.25,0.00,0.75,18.6,148.0,14.00,...,86.0,167,74,56.0,0.622222,0.510638,73,86,1,76
6,dried debris,22.5,0.018605,0.55,0.00,0.25,0.75,18.6,106.0,14.30,...,89.0,171,75,11.0,0.122222,NaN,NaN,NaN,2,19
7,NaN,22.5,0.018605,0.55,0.00,0.25,0.75,18.6,138.0,14.15,...,87.5,173,76,44.5,0.494444,NaN,NaN,NaN,2,38


## Attaching Image Paths to the Samples

In [4]:
def find_photo(folder, pic_num):

    for ext in ["jpg","JPG","jpeg","JPEG","png","PNG"]:
        for padding in [3,4]:
            pattern = str(folder / f"*{pic_num:0{padding}d}.{ext}")
            matches = glob.glob(pattern)

            if matches:
                return matches[0]

    raise FileNotFoundError(f"No photo for {pic_num}")


In [5]:
samples["top_path"] = samples["pic_top"].apply(lambda x: find_photo(TOP_FOLDER, x))
samples["side_path"] = samples["pic_side"].apply(lambda x: find_photo(SIDE_FOLDER, x))
display(samples[["exp_id","volume","top_path","side_path"]].head())

,exp_id,volume,top_path,side_path
2,1,38,top_view_images/P2090072.JPG,side_view_images/P2090161.JPG
3,1,57,top_view_images/P2090073.JPG,side_view_images/P2090163.JPG
4,1,76,top_view_images/P2090074.JPG,side_view_images/P2090167.JPG
6,2,19,top_view_images/P2090075.JPG,side_view_images/P2090171.JPG
7,2,38,top_view_images/P2090076.JPG,side_view_images/P2090173.JPG


## Feature-extraction Model Setup

In [6]:
weights = models.ResNet50_Weights.DEFAULT
preprocess = weights.transforms()
resnet = models.resnet50(weights=weights)
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


## Embedding Computation

In [7]:
def embedding_cache_path(image_path):
    key = str(Path(image_path).resolve())
    h = hashlib.md5(key.encode()).hexdigest()
    return EMBEDDING_CACHE_DIR / f"{h}.npy"

@torch.no_grad()
def compute_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    x = preprocess(image).unsqueeze(0).to(DEVICE)
    feat = feature_extractor(x)
    feat = feat.flatten(1).cpu().numpy()[0]
    return feat.astype(np.float32)

def get_embedding(image_path):
    cache_file = embedding_cache_path(image_path)

    # If the embedding is already cached, use that
    if cache_file.exists():
        return np.load(cache_file)

    feat = compute_embedding(image_path)
    np.save(cache_file, feat)
    return feat

In [8]:
top_embeddings = []
side_embeddings = []

for _, row in tqdm(samples.iterrows(), total=len(samples)):
    top_vector = get_embedding(row["top_path"])
    side_vector = get_embedding(row["side_path"])
    top_embeddings.append(top_vector)
    side_embeddings.append(side_vector)

top_embeddings = np.vstack(top_embeddings)
side_embeddings = np.vstack(side_embeddings)

X = np.hstack([top_embeddings, side_embeddings])
y = samples["volume"].values
groups = samples["exp_id"].values

print("X shape:", X.shape)

100%|██████████| 140/140 [00:00<00:00, 289.65it/s]

X shape: (140, 4096)


## Model Configuration & Hyperparameter Grids

In [9]:
def make_pipeline(estimator):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler",  StandardScaler()),
        ("model",   estimator),
    ])

# Each entry: (pipeline, param_grid)
MODEL_CONFIGS = {
    "dummy_mean": (
        DummyRegressor(strategy="mean"),
        {}, 
    ),

    "ridge": (
        make_pipeline(Ridge()),
        {"model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]},
    ),

    "elasticnet": (
        make_pipeline(ElasticNet(max_iter=20000)),
        {
            "model__alpha":    [0.0001, 0.001, 0.01, 0.1, 1.0],
            "model__l1_ratio": [0.1, 0.2, 0.5, 0.8, 0.9],
        },
    ),

    "random_forest": (
        # No scaler needed for trees, but consistency yk
        Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model",   RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        {   # removed some since my kernel couldnt handle it and I got impatiant
            "model__n_estimators":    [100, 300], # cut 500
            "model__max_depth":       [None, 10], # cut 5, 20
            "model__min_samples_leaf":[1, 2], # cut 4
            "model__max_features":    ["sqrt", 0.3], # cut 0.5
        },
    ),

    "extra_trees": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model",   ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        {   # same here
            "model__n_estimators":    [100, 300],
            "model__max_depth":       [None, 10],
            "model__min_samples_leaf":[1, 2],
            "model__max_features":    ["sqrt", 0.3],
        },
    ),
}

print("Grid sizes per model:")
from sklearn.model_selection import ParameterGrid
for name, (pipe, grid) in MODEL_CONFIGS.items():
    print(f"  {name:20s}: {len(list(ParameterGrid(grid))):4d} combinations")


Grid sizes per model:
  dummy_mean          :    1 combinations
  ridge               :    7 combinations
  elasticnet          :   25 combinations
  random_forest       :   16 combinations
  extra_trees         :   16 combinations


## Nested Cross-Validation

```
outer loop  ->  GroupKFold(5, group=exp_id)  ->  unbiased performance estimate
  |--- inner loop  ->  GridSearchCV(GroupKFold(5), group=exp_id)  ->  best hyperparameters
```

In [ ]:
outer_cv = GroupKFold(n_splits=OUTER_SPLITS)

nested_results  = {}   # model_name -> list[dict]  (one dict per outer fold)
oof_predictions = {}   # model_name -> np.ndarray  (out-of-fold predictions)

for model_name, (pipeline, param_grid) in MODEL_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")

    fold_records = []
    oof_pred = np.full(len(y), np.nan)

    for fold_idx, (train_idx, test_idx) in enumerate(
            outer_cv.split(X, y, groups)):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        g_train         = groups[train_idx]

        # Inner CV: hyperparameter search ─────────────────────────────────
        if param_grid:
            n_inner  = min(INNER_SPLITS, len(np.unique(g_train)))
            inner_cv = GroupKFold(n_splits=n_inner)

            gs = GridSearchCV(
                estimator  = sklearn.clone(pipeline),
                param_grid = param_grid,
                cv         = inner_cv,
                scoring    = "neg_mean_absolute_error",
                refit      = True,   # refit on full X_train with best params
                n_jobs     = -1,
            )
            gs.fit(X_train, y_train, groups=g_train)

            best_estimator = gs.best_estimator_
            best_params    = gs.best_params_
            inner_mae      = -gs.best_score_
        else:
            # DummyRegressor: nothing to tune
            best_estimator = sklearn.clone(pipeline)
            best_estimator.fit(X_train, y_train)
            best_params    = {}
            inner_mae      = None

        # Evaluate on held-out outer fold ──────────────────────────────────
        y_pred = best_estimator.predict(X_test)
        oof_pred[test_idx] = y_pred

        mae  = mean_absolute_error(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)

        fold_records.append({
            "fold":        fold_idx + 1,
            "best_params": best_params,
            "inner_MAE":   inner_mae,
            "MAE":         mae,
            "RMSE":        rmse,
            "R2":          r2,
        })
        print(f"  Fold {fold_idx+1}:  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.3f}"
              + (f"  →  {best_params}" if best_params else ""))

    nested_results[model_name]  = fold_records
    oof_predictions[model_name] = oof_pred

print("\nNested CV complete.")



  dummy_mean
  Fold 1:  MAE=19.00  RMSE=21.24  R2=0.000
  Fold 2:  MAE=19.00  RMSE=21.24  R2=0.000
  Fold 3:  MAE=19.00  RMSE=21.25  R2=-0.001
  Fold 4:  MAE=18.66  RMSE=20.94  R2=-0.004
  Fold 5:  MAE=19.35  RMSE=21.56  R2=-0.000

  ridge


In [ ]:
rows = []
for name, folds in nested_results.items():
    rows.append({
        "model":       name,
        "cv_mae_mean": np.mean([f["MAE"]  for f in folds]),
        "cv_mae_std":  np.std( [f["MAE"]  for f in folds]),
        "cv_mse_mean": np.mean([f["RMSE"] for f in folds])**2,
        "cv_mse_std":  np.std( [f["RMSE"] for f in folds]),
        "cv_rmse_mean":np.mean([f["RMSE"] for f in folds]),
        "cv_rmse_std": np.std( [f["RMSE"] for f in folds]),
        "cv_r2_mean":  np.mean([f["R2"]   for f in folds]),
        "cv_r2_std":   np.std( [f["R2"]   for f in folds]),
    })

results_df = (
    pd.DataFrame(rows)
    .sort_values("cv_mae_mean")
    .reset_index(drop=True)
)
display(results_df)


In [ ]:
print("Best hyperparameters selected per outer fold")
print("(stability across folds shows how sensitive the model is to training data)\n")

for name, folds in nested_results.items():
    if not any(f["best_params"] for f in folds):
        continue
    print(f"{name}")
    for f in folds:
        print(f"  Fold {f['fold']}: {f['best_params']}")
    print()


## Out-of-fold Output Analysis

In [ ]:
# Out-of-fold = for each data point we calculate the predicted volume,
# using a model trained on the folds that did not contain that point

rows = []
for name, folds in nested_results.items():
    pred = oof_predictions[name]

    # Skip rows where OOF prediction wasn't filled (just in case)
    valid = ~np.isnan(pred)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # True vs predicted volume
    axes[0].scatter(y[valid], pred[valid], alpha=0.8)
    axes[0].set_xlim(0, 100)
    axes[0].set_ylim(0, 100)
    axes[0].plot([y.min(), y.max()], [y.min(), y.max()], "--")
    axes[0].set_xlabel("True volume")
    axes[0].set_ylabel("Predicted volume")
    axes[0].set_title(f"Out-of-fold Predictions — {name}")

    # Histogram of predictions
    axes[1].hist(pred[valid], bins=20, range=(0, 100))
    axes[1].set_xlim(0, 100)
    axes[0].set_ylim(0, 100)
    axes[1].set_xlabel("Predicted Volume")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Prediction Distribution — {name}")

    plt.tight_layout()
    plt.show()

    rows.append({
        "model": name,
        "mae":   mean_absolute_error(y[valid], pred[valid]),
        "mse":   mean_squared_error(y[valid], pred[valid]),
        "rmse":  root_mean_squared_error(y[valid], pred[valid]),
        "r2":    r2_score(y[valid], pred[valid]),
    })

results_df = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
display(results_df)


## Leave-one-volume-out Cross-validation

In [ ]:
unique_volumes  = np.sort(np.unique(y))
fold_rows       = []
all_predictions = []

for model_name, (pipeline, param_grid) in MODEL_CONFIGS.items():

    for held_out_volume in unique_volumes:
        train_mask = y != held_out_volume
        test_mask  = y == held_out_volume

        X_train, y_train = X[train_mask], y[train_mask]
        X_test,  y_test  = X[test_mask],  y[test_mask]
        g_train          = groups[train_mask]
        g_test           = groups[test_mask]

        # Inner CV on training volumes
        if param_grid:
            n_inner  = min(INNER_SPLITS, len(np.unique(g_train)))
            inner_cv = GroupKFold(n_splits=n_inner)
            gs = GridSearchCV(
                sklearn.clone(pipeline), param_grid,
                cv=inner_cv, scoring="neg_mean_absolute_error",
                refit=True, n_jobs=-1,
            )
            gs.fit(X_train, y_train, groups=g_train)
            final_model = gs.best_estimator_
            best_params = gs.best_params_
        else:
            final_model = sklearn.clone(pipeline)
            final_model.fit(X_train, y_train)
            best_params = {}

        y_pred = final_model.predict(X_test)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)

        fold_rows.append({
            "model":           model_name,
            "held_out_volume": float(held_out_volume),
            "pred_mean":       float(np.mean(y_pred)),
            "pred_std":        float(np.std(y_pred)),
            "best_params":     str(best_params),
            "mae":             mae,
            "mse":             mse,
            "rmse":            rmse,
            "n_train":         len(y_train),
            "n_test":          len(y_test),
        })

        all_predictions.append(pd.DataFrame({
            "model":           model_name,
            "held_out_volume": held_out_volume,
            "exp_id":          g_test,
            "true_volume":     y_test,
            "pred_volume":     y_pred,
            "abs_error":       np.abs(y_test - y_pred),
        }))

leave_one_volume_results     = pd.DataFrame(fold_rows)
leave_one_volume_predictions = pd.concat(all_predictions, ignore_index=True)

display(leave_one_volume_results.drop(columns="best_params"))

In [ ]:
summary_rows = []
for model_name, g in leave_one_volume_results.groupby("model"):
    summary_rows.append({
        "model":     model_name,
        "mean_mae":  g["mae"].mean(),
        "std_mae":   g["mae"].std(),
        "mean_mse":  g["mse"].mean(),
        "std_mse":   g["mse"].std(),
        "mean_rmse": g["rmse"].mean(),
        "std_rmse":  g["rmse"].std(),
    })

leave_one_volume_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("mean_mae")
    .reset_index(drop=True)
)

print("Average performance and deviation across all held-out volumes:")
display(leave_one_volume_summary)


In [ ]:
volume_summary = (
    leave_one_volume_results
    .pivot_table(
        index="held_out_volume",
        columns="model",
        values="mae"
    )
)

print("MAE of each model on each held-out volume:")
display(volume_summary)

In [ ]:
print("Predicted volumes on the held-out sets, for each model:")

for model_name in MODEL_CONFIGS:
    predictions = leave_one_volume_predictions[
        leave_one_volume_predictions["model"] == model_name
    ].copy()

    fig, axes = plt.subplots(
        1, len(unique_volumes),
        figsize=(4 * len(unique_volumes), 4),
        sharey=True,
    )
    if len(unique_volumes) == 1:
        axes = [axes]

    for ax, vol in zip(axes, unique_volumes):
        subset = predictions[predictions["held_out_volume"] == vol]
        ax.hist(subset["pred_volume"], bins=20, range=(0, 100))
        ax.set_xlim(0, 100)
        ax.axvline(vol, linestyle="--")
        ax.set_title(f"Held out {int(vol)}")
        ax.set_xlabel("Predicted volume")

    axes[0].set_ylabel(f"Count — {model_name}")
    plt.tight_layout()
    plt.show()
